In [ ]:
SEED = 42
import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)


# Task 01 - Data Ingestion (NYC Airbnb 2019)
Goal: clean and validate data for downstream modelling of `price` without transforming the target yet.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_PATH = Path("../../../data/raw/AB_NYC_2019.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
df.head()

## Cleaning Decisions and Justification
1. `host_id` was dropped because it is an identifier and does not encode stable listing characteristics for generalization.
2. `last_review` was dropped because ~20.56% missing values create sparse date information at this stage, and date engineering belongs in later modelling tasks.
3. `name` and `host_name` missing values were filled with `Unknown` to preserve rows while keeping string columns complete.
4. `reviews_per_month` missing values were filled with `0` because in this dataset they correspond to listings with no reviews, so zero is semantically meaningful rather than arbitrary imputation.
5. Rows with `price <= 0` were removed as invalid target records (data quality issue).
6. `minimum_nights` was clipped at the 99th percentile to reduce leverage from extreme listing constraints while retaining most observations.
7. High `price` values were **not transformed** in Task 01 to respect project constraints; any target transformation is deferred to Task 03.

In [ ]:
clean_df = df.copy()
clean_df = clean_df.drop(columns=["host_id", "last_review"])
clean_df["name"] = clean_df["name"].fillna("Unknown")
clean_df["host_name"] = clean_df["host_name"].fillna("Unknown")
clean_df["reviews_per_month"] = clean_df["reviews_per_month"].fillna(0.0)
clean_df = clean_df[clean_df["price"] > 0].copy()
cap = clean_df["minimum_nights"].quantile(0.99)
clean_df["minimum_nights"] = np.minimum(clean_df["minimum_nights"], cap)
clean_df.isna().sum().sum()

## Output Files
- `outputs/schema_log.txt`
- `outputs/missingness_report.csv`
- `outputs/outlier_flags.txt`
- `outputs/ingested.csv`